# Dev — Week 5–7: TCR Analysis, Regression, and ES Comparison
**Author:** Dev | **Branch:** `feature/risk-metrics`  
**Inputs:** Nihan's CSVs in `/data/`, Robert's backtesting split (years 1–4 train, year 5 holdout)  
**Outputs:** `/results/tcr_results.csv`, `/results/tcr_vs_kurtosis_scatter.png`, `/results/es_comparison_table.csv`

---

## Purpose

This notebook delivers all of Dev's Week 5–7 tasks:

| # | Task | Section |
|---|------|---------|
| 1 | TCR at 95% and 99% for all 5 equities using backtesting split | §2 |
| 2 | Cross-sectional scatter: TCR vs excess kurtosis κₑ | §3 |
| 3 | Linear regression of TCR on κₑ — slope, R², p-value | §4 |
| 4 | GBM-simulated ES vs historically realized average tail losses | §5 |
| 5 | ES ratio (GBM simulated / realized) per equity | §5 |

Run **Kernel → Restart & Run All** to fully reproduce from a fresh state.

## 0. Imports and Configuration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from scipy import stats
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

NOTEBOOK_DIR     = Path().resolve()
REPO_ROOT        = NOTEBOOK_DIR.parents[1]
DATA_DIR         = REPO_ROOT / 'data'
RESULTS_DIR      = REPO_ROOT / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

TICKERS           = ['AAPL', 'AMZN', 'GOOGL', 'MSFT', 'TSLA']
CONFIDENCE_LEVELS = [0.95, 0.99]
TRADING_DAYS      = 252
N_PATHS           = 10_000

print(f'Repo root : {REPO_ROOT}')
print(f'Data dir  : {DATA_DIR}')

## 1. Data Loading and Backtesting Split

Following Robert's backtesting convention, we split each equity's 5-year window:
- **Training period (years 1–4):** used to calibrate GBM parameters $\hat{\mu}$ and $\hat{\sigma}$
- **Holdout period (year 5):** used to count VaR breaches and evaluate out-of-sample performance

This mirrors the split in Robert's backtesting script so that TCR values here are directly
comparable to his `/results/tcr_results.csv` output.

In [ ]:
prices      = {}
log_returns = {}
lr_train    = {}
lr_holdout  = {}

for t in TICKERS:
    csv_path = DATA_DIR / f'{t}_daily_5y.csv'
    if not csv_path.exists():
        raise FileNotFoundError(f'{csv_path} not found — ensure feature/risk-metrics is rebased onto dev.')
    df  = pd.read_csv(csv_path, index_col='Date', parse_dates=True)
    col = 'Adj Close' if 'Adj Close' in df.columns else 'Close'
    p   = df[col].dropna()
    prices[t]      = p
    log_returns[t] = np.log(p / p.shift(1)).dropna()

    # Backtesting split: last TRADING_DAYS rows = holdout year 5
    total       = len(p)
    holdout_n   = TRADING_DAYS
    train_p     = p.iloc[:total - holdout_n]
    holdout_p   = p.iloc[total - holdout_n:]
    lr_train[t]   = np.log(train_p / train_p.shift(1)).dropna().values.flatten()
    lr_holdout[t] = np.log(holdout_p / holdout_p.shift(1)).dropna().values.flatten()

    print(f'{t}: {total} obs | train={len(lr_train[t])} | holdout={len(lr_holdout[t])}')

print('\nAll five equities loaded and split.')

## 1b. GBM Calibration and Simulation

### Calibration Formula

From the training log-returns $\{r_1, \ldots, r_T\}$:

$$\hat{\sigma} = s_r \times \sqrt{252}, \qquad
\hat{\mu} = \bar{r} \times 252 + \frac{\hat{\sigma}^2}{2}$$

The GBM daily log-return under the calibrated model follows:

$$r_t^{\text{sim}} \sim \mathcal{N}\!\left[\left(\hat{\mu} - \frac{\hat{\sigma}^2}{2}\right)\Delta t,\
\hat{\sigma}^2 \Delta t\right]$$

We simulate 10,000 paths × 252 steps = 2,520,000 daily log-return draws per equity,
then extract the implied VaR and ES from the simulated daily distribution.

In [ ]:
def calibrate(lr_array):
    """Return (mu_ann, sigma_ann) calibrated from daily log-return array."""
    daily_mean = float(np.mean(lr_array))
    daily_std  = float(np.std(lr_array, ddof=1))
    sigma = daily_std * np.sqrt(TRADING_DAYS)
    mu    = daily_mean * TRADING_DAYS + 0.5 * sigma**2
    return mu, sigma

def simulate_daily_lr(mu, sigma, paths=N_PATHS, N=TRADING_DAYS, seed=42):
    """Simulate (paths × N) daily GBM log-returns; return flattened 1-D array."""
    rng  = np.random.default_rng(seed)
    dt   = 1.0 / N
    Z    = rng.normal(size=(paths, N))
    return ((mu - 0.5*sigma**2)*dt + sigma*np.sqrt(dt)*Z).flatten()

def var_q(returns, cl):
    return float(-np.quantile(np.asarray(returns).flatten(), 1 - cl))

def es_q(returns, cl):
    r = np.asarray(returns).flatten()
    tail = r[r < -var_q(r, cl)]
    return float(-tail.mean()) if len(tail) > 0 else np.nan


# Calibrate and simulate all equities
gbm_params   = {}  # (mu, sigma) per ticker
sim_daily_lr = {}  # simulated daily log-returns per ticker

print('Calibrated GBM parameters (training data):')  
print(f'{"Ticker":6}  {"mu_ann":>8}  {"sigma_ann":>10}  {"Ann.Vol":>8}')
print('-' * 40)
for t in TICKERS:
    mu, sigma = calibrate(lr_train[t])
    gbm_params[t]   = (mu, sigma)
    sim_daily_lr[t] = simulate_daily_lr(mu, sigma, seed=42)
    print(f'{t:6}  {mu:>8.4f}  {sigma:>10.4f}  {sigma:>7.2%}')

---
## 2. Tail Coverage Ratio (TCR) — Backtested

### Definition

The **Tail Coverage Ratio** quantifies how well the GBM-predicted VaR threshold captures
realized tail events in the out-of-sample holdout period:

$$\text{TCR}_\alpha = \frac{\text{Realized breach frequency}}{\text{Theoretical breach frequency}}
= \frac{\#\{r_t^{\text{holdout}} < -\widehat{\text{VaR}}_\alpha^{\text{GBM}}\} / n_{\text{holdout}}}{1-\alpha}$$

- **TCR = 1.0** → GBM is correctly calibrated (realized breaches match prediction)
- **TCR > 1.0** → GBM *underestimates* tail risk (more breaches than predicted — fat tails)
- **TCR < 1.0** → GBM *overestimates* tail risk (fewer breaches — holdout was a calm period)

This is computed from Robert's backtesting split: GBM calibrated on years 1–4, breaches
counted in year 5.

In [ ]:
tcr_records = []

for t in TICKERS:
    mu, sigma = gbm_params[t]
    holdout   = lr_holdout[t]
    n_holdout = len(holdout)

    for cl in CONFIDENCE_LEVELS:
        gbm_var       = var_q(sim_daily_lr[t], cl)
        gbm_es        = es_q(sim_daily_lr[t], cl)
        hist_var_hold = var_q(holdout, cl)
        hist_es_hold  = es_q(holdout, cl)

        n_breach      = int(np.sum(holdout < -gbm_var))
        realized_freq = n_breach / n_holdout
        theo_freq     = 1 - cl
        tcr           = realized_freq / theo_freq

        tcr_records.append({
            'Ticker'         : t,
            'Conf.'          : f'{cl:.0%}',
            'mu_ann'         : round(mu, 4),
            'sigma_ann'      : round(sigma, 4),
            'GBM VaR'        : f'{gbm_var:.3%}',
            'Hist VaR (hold)': f'{hist_var_hold:.3%}',
            'Breaches'       : n_breach,
            'n_holdout'      : n_holdout,
            'Realized freq'  : f'{realized_freq:.3%}',
            'Theo. freq'     : f'{theo_freq:.1%}',
            'TCR'            : round(tcr, 4),
            'GBM overestimates?': 'NO (calm holdout)' if tcr < 1 else 'YES (fat tails)',
        })

tcr_df = pd.DataFrame(tcr_records)

print('=' * 100)
print('  TCR Results — Train: Years 1–4 | Holdout: Year 5')
print('=' * 100)
print(tcr_df[['Ticker','Conf.','GBM VaR','Hist VaR (hold)',
               'Breaches','n_holdout','Realized freq','Theo. freq','TCR',
               'GBM overestimates?']].to_string(index=False))
print('=' * 100)

# Save to /results/
csv_out = RESULTS_DIR / 'tcr_results.csv'
tcr_df.to_csv(csv_out, index=False)
print(f'\nSaved → {csv_out.name}')
tcr_df

---
## 3. Cross-Sectional Scatter Plot: TCR vs Excess Kurtosis

### Hypothesis

The central claim of the paper is that **TCR correlates positively with excess kurtosis** $\kappa_e$:
equities with fatter tails relative to normal should show higher TCR because GBM's normality
assumption systematically misses more tail events as kurtosis rises.

The scatter below plots TCR against $\kappa_e$ for all five equities at both confidence levels.
Each point is one (equity, confidence level) observation. The regression line is fitted separately
at 95% and 99%.

In [ ]:
# Compute excess kurtosis for full log-return series (all 5 years)
kurtosis = {t: float(log_returns[t].kurt()) for t in TICKERS}
skewness = {t: float(log_returns[t].skew()) for t in TICKERS}

print('Excess kurtosis (full sample):')
for t in TICKERS:
    print(f'  {t}: κe = {kurtosis[t]:.4f}  |  skew = {skewness[t]:.4f}')

# Organise TCR values by confidence level
tcr_by_cl = {}
for cl in CONFIDENCE_LEVELS:
    tcr_by_cl[cl] = {row['Ticker']: row['TCR']
                     for row in tcr_records if row['Conf.'] == f'{cl:.0%}'}

# ── Plot ──
fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=False)
fig.patch.set_facecolor('#f9f9f9')

colors95 = ['#1e3a5f','#2e86ab','#a23b72','#f18f01','#c73e1d']
colors99 = colors95

for ax_i, (cl, ax) in enumerate(zip(CONFIDENCE_LEVELS, axes)):
    ax.set_facecolor('#f9f9f9')
    kurt_vals = [kurtosis[t] for t in TICKERS]
    tcr_vals  = [tcr_by_cl[cl][t] for t in TICKERS]

    # Scatter
    for i, t in enumerate(TICKERS):
        ax.scatter(kurtosis[t], tcr_by_cl[cl][t],
                   color=colors95[i], s=120, zorder=4, label=t)
        ax.annotate(t, (kurtosis[t], tcr_by_cl[cl][t]),
                    textcoords='offset points', xytext=(6, 4),
                    fontsize=9, color=colors95[i], fontweight='bold')

    # OLS regression line
    slope, intercept, r_val, p_val, se = stats.linregress(kurt_vals, tcr_vals)
    x_fit = np.linspace(min(kurt_vals)*0.9, max(kurt_vals)*1.05, 100)
    ax.plot(x_fit, slope*x_fit + intercept,
            color='#555', lw=1.8, ls='--', alpha=0.7,
            label=f'OLS: slope={slope:.4f}, R²={r_val**2:.3f}')

    # TCR = 1 reference line
    ax.axhline(1.0, color='#e74c3c', lw=1.2, ls=':', alpha=0.7, label='TCR = 1.0 (ideal)')

    # Annotation box
    stats_txt = (f'slope  = {slope:.4f}\n'
                 f'R²     = {r_val**2:.4f}\n'
                 f'p-val  = {p_val:.4f}\n'
                 f'SE     = {se:.4f}')
    ax.text(0.97, 0.97, stats_txt, transform=ax.transAxes,
            fontsize=8.5, va='top', ha='right',
            bbox=dict(boxstyle='round,pad=0.4', facecolor='white', alpha=0.88))

    ax.set_title(f'TCR vs Excess Kurtosis — {int(cl*100)}% Confidence',
                 fontsize=12, fontweight='bold')
    ax.set_xlabel('Excess Kurtosis $\\kappa_e$ (full sample)', fontsize=11)
    ax.set_ylabel(f'TCR at {int(cl*100)}%', fontsize=11)
    ax.legend(fontsize=8.5, framealpha=0.85, loc='upper left')
    ax.grid(alpha=0.3)

fig.suptitle('Cross-Sectional TCR vs Excess Kurtosis\n'
             'GBM calibrated on years 1–4 | Breaches counted in holdout year 5',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()

fig_path = RESULTS_DIR / 'tcr_vs_kurtosis_scatter.png'
fig.savefig(fig_path, dpi=300, bbox_inches='tight')
print(f'Saved → {fig_path.name}')
plt.show()

---
## 4. Linear Regression: TCR ~ Excess Kurtosis

### Statistical Model

We fit the cross-sectional OLS regression:

$$\text{TCR}_{\alpha,i} = \beta_0 + \beta_1 \kappa_{e,i} + \varepsilon_i, \quad i \in \{\text{AAPL, AMZN, GOOGL, MSFT, TSLA}\}$$

where $\kappa_{e,i}$ is the full-sample excess kurtosis of equity $i$.  
This is the **quantitative claim** of the paper: TCR is predictable from kurtosis.

We report the slope $\hat{\beta}_1$, $R^2$, and $p$-value for the slope at both confidence levels.
A positive slope confirms that higher-kurtosis equities have more tail breaches relative
to GBM's prediction.

In [ ]:
reg_records = []

print('=== Linear Regression: TCR ~ Excess Kurtosis ===')
print(f'{"Conf":6} {"Slope":>8} {"Intercept":>11} {"R²":>8} {"p-value":>10} {"Std Err":>10}')
print('-' * 60)

for cl in CONFIDENCE_LEVELS:
    kurt_vals = [kurtosis[t] for t in TICKERS]
    tcr_vals  = [tcr_by_cl[cl][t] for t in TICKERS]

    slope, intercept, r_val, p_val, se = stats.linregress(kurt_vals, tcr_vals)
    r2 = r_val**2

    print(f'{int(cl*100):>4}%  {slope:>8.4f}  {intercept:>11.4f}  {r2:>8.4f}  {p_val:>10.4f}  {se:>10.4f}')

    reg_records.append({
        'Confidence Level': f'{int(cl*100)}%',
        'Slope (β₁)':      round(slope, 6),
        'Intercept (β₀)':  round(intercept, 6),
        'R²':              round(r2, 6),
        'p-value (slope)': round(p_val, 6),
        'Std Error':       round(se, 6),
        'n':               len(TICKERS),
        'Significant (5%)?': 'Yes' if p_val < 0.05 else 'No',
    })

reg_df = pd.DataFrame(reg_records).set_index('Confidence Level')
print()
print('Regression Summary Table:')
display(reg_df)

print()
print('Interpretation:')
for cl in CONFIDENCE_LEVELS:
    row = [r for r in reg_records if r['Confidence Level']==f'{int(cl*100)}%'][0]
    sig = row['Significant (5%)?']
    print(f'  {int(cl*100)}%: slope={row["Slope (β₁)"]:.4f}, R²={row["R²"]:.4f}, '
          f'p={row["p-value (slope)"]:.4f} — {sig} at 5% level')
    if row['Slope (β₁)'] > 0:
        print(f'       Positive slope confirms: higher κe → higher TCR → GBM underestimates more')

reg_df.to_csv(RESULTS_DIR / 'tcr_regression_results.csv')
print('\nSaved → tcr_regression_results.csv')

---
## 5. ES Comparison: GBM-Simulated vs Historically Realized Tail Losses

### Motivation

VaR tells us the *threshold* below which bad outcomes fall; ES tells us the *average loss*
in the tail beyond that threshold. Here we compare:

| Quantity | Source | Description |
|----------|--------|-------------|
| $\widehat{\text{ES}}_\alpha^{\text{GBM}}$ | 10k GBM paths (daily) | ES implied by GBM normality assumption |
| $\widehat{\text{ES}}_\alpha^{\text{realized}}$ | Holdout year 5 | Average of actual returns below GBM VaR threshold |
| **ES Ratio** | GBM / Realized | Ratio > 1 → GBM overstates conditional tail severity |

### ES Ratio Formula

$$\text{ES Ratio}_\alpha = \frac{\widehat{\text{ES}}_\alpha^{\text{GBM}}}{\widehat{\text{ES}}_\alpha^{\text{realized}}^{\text{holdout}}}$$

An ES Ratio > 1.0 means GBM's simulated tail losses are larger than what was actually
experienced in the holdout year — which may seem counter-intuitive given fat tails,
but can occur in a relatively calm holdout period. A ratio < 1.0 means realized tail
losses were more severe than GBM predicted.

In [ ]:
es_records = []

for t in TICKERS:
    holdout = lr_holdout[t]
    for cl in CONFIDENCE_LEVELS:
        gbm_var  = var_q(sim_daily_lr[t], cl)
        gbm_es   = es_q(sim_daily_lr[t], cl)

        # Realized average tail loss: holdout returns below GBM VaR threshold
        tail_holdout  = holdout[holdout < -gbm_var]
        realized_es   = float(-np.mean(tail_holdout)) if len(tail_holdout) > 0 else np.nan

        # Historical ES on holdout (using its own quantile)
        hist_es_hold  = es_q(holdout, cl)

        es_ratio = gbm_es / realized_es if realized_es and not np.isnan(realized_es) else np.nan

        es_records.append({
            'Ticker'              : t,
            'Conf.'               : f'{cl:.0%}',
            'GBM VaR threshold'   : f'{gbm_var:.3%}',
            'GBM ES (simulated)'  : f'{gbm_es:.3%}',
            'Realized ES (holdout)':f'{realized_es:.3%}' if not np.isnan(realized_es) else 'N/A',
            'n_tail_obs'          : len(tail_holdout),
            'Hist ES (holdout)'   : f'{hist_es_hold:.3%}',
            'ES Ratio (GBM/Real)' : round(es_ratio, 4) if not np.isnan(es_ratio) else 'N/A',
            'GBM overstates ES?'  : ('Yes (GBM > Realized)' if not np.isnan(es_ratio) and es_ratio > 1
                                     else 'No  (Realized > GBM)'),
        })

es_df = pd.DataFrame(es_records)

print('=' * 110)
print('  ES Comparison — GBM Simulated vs Holdout Realized')
print('=' * 110)
print(es_df.to_string(index=False))
print('=' * 110)

es_df.to_csv(RESULTS_DIR / 'es_comparison_table.csv', index=False)
print('\nSaved → es_comparison_table.csv')
es_df

## 5b. ES Comparison Bar Chart

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.patch.set_facecolor('#f9f9f9')

for ax_i, (cl, ax) in enumerate(zip(CONFIDENCE_LEVELS, axes)):
    ax.set_facecolor('#f9f9f9')
    rows = [r for r in es_records if r['Conf.'] == f'{cl:.0%}']
    tickers   = [r['Ticker'] for r in rows]
    gbm_es_v  = [float(r['GBM ES (simulated)'].strip('%'))/100 for r in rows]
    real_es_v = [float(r['Realized ES (holdout)'].strip('%'))/100
                 if r['Realized ES (holdout)'] != 'N/A' else 0 for r in rows]

    x = np.arange(len(tickers))
    w = 0.35
    b1 = ax.bar(x - w/2, gbm_es_v,  w, label='GBM Simulated ES', color='#1e3a5f', alpha=0.85)
    b2 = ax.bar(x + w/2, real_es_v, w, label='Realized ES (holdout)', color='#e74c3c', alpha=0.85)

    ax.set_xticks(x); ax.set_xticklabels(tickers, fontsize=11)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=1))
    ax.set_title(f'GBM Simulated vs Realized ES — {int(cl*100)}% Confidence',
                 fontsize=12, fontweight='bold')
    ax.set_ylabel('Expected Shortfall (daily log-return)', fontsize=10)
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)

    # ES Ratio labels
    for i, row in enumerate(rows):
        ratio = row['ES Ratio (GBM/Real)']
        if ratio != 'N/A':
            ax.text(x[i], max(gbm_es_v[i], real_es_v[i]) + 0.0008,
                    f'ratio\n{ratio:.3f}', ha='center', fontsize=7.5, color='#333')

plt.suptitle('Expected Shortfall: GBM Model vs Out-of-Sample Realized\n'
             'ES Ratio = GBM ES / Realized ES (holdout year 5)',
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()

fig_path = RESULTS_DIR / 'es_comparison_bar.png'
fig.savefig(fig_path, dpi=300, bbox_inches='tight')
print(f'Saved → {fig_path.name}')
plt.show()

---
## 6. Complete Summary and Interpretation

The table below consolidates all Week 5–7 deliverables into one paper-ready summary.

In [ ]:
summary_rows = []
for t in TICKERS:
    for cl in CONFIDENCE_LEVELS:
        tcr_row = next(r for r in tcr_records if r['Ticker']==t and r['Conf.']==f'{cl:.0%}')
        es_row  = next(r for r in es_records  if r['Ticker']==t and r['Conf.']==f'{cl:.0%}')
        summary_rows.append({
            'Ticker'           : t,
            'Conf.'            : f'{cl:.0%}',
            'Excess Kurt. (κe)': round(kurtosis[t], 4),
            'TCR'              : tcr_row['TCR'],
            'GBM ES'           : es_row['GBM ES (simulated)'],
            'Realized ES'      : es_row['Realized ES (holdout)'],
            'ES Ratio'         : es_row['ES Ratio (GBM/Real)'],
            'VaR Breaches'     : f"{tcr_row['Breaches']}/{tcr_row['n_holdout']}",
        })

summary_df = pd.DataFrame(summary_rows)
print('=' * 90)
print('  Complete Week 5–7 Summary — TCR, Kurtosis, ES Comparison')
print('=' * 90)
print(summary_df.to_string(index=False))
print('=' * 90)
summary_df.to_csv(RESULTS_DIR / 'week57_summary.csv', index=False)
print('\nSaved → week57_summary.csv')

print('\n--- Regression Takeaway ---')
for cl in CONFIDENCE_LEVELS:
    kurt_vals = [kurtosis[t] for t in TICKERS]
    tcr_vals  = [tcr_by_cl[cl][t] for t in TICKERS]
    slope, intercept, r_val, p_val, se = stats.linregress(kurt_vals, tcr_vals)
    print(f'  {int(cl*100)}%: TCR = {intercept:.4f} + {slope:.4f}×κe  '
          f'| R²={r_val**2:.4f} | p={p_val:.4f}')
    
summary_df

---
## 7. Reproducibility Checklist

- [ ] `feature/risk-metrics` is on latest `dev` HEAD
- [ ] `Kernel → Restart & Run All` — zero errors
- [ ] `results/tcr_results.csv` — TCR for all 5 equities × 2 confidence levels
- [ ] `results/tcr_vs_kurtosis_scatter.png` — scatter + OLS lines at both levels
- [ ] `results/es_comparison_bar.png` — GBM vs realized ES bar charts
- [ ] `results/es_comparison_table.csv` — ES ratio per equity per confidence level
- [ ] `results/tcr_regression_results.csv` — slope, R², p-value
- [ ] `results/week57_summary.csv` — consolidated table
- [ ] PR updated into `dev` with Week 5–7 commit

---

## References

- Artzner, P., Delbaen, F., Eber, J.-M., & Heath, D. (1999). Coherent measures of risk. *Mathematical Finance*, 9(3), 203–228.
- Hull, J. C. (2018). *Risk Management and Financial Institutions* (5th ed.). Wiley.
- McNeil, A. J., Frey, R., & Embrechts, P. (2015). *Quantitative Risk Management* (Rev. ed.). Princeton UP.